# Couche -> Gold

Construire les tables d'analyse à partir des fichiers du [notebook Silver](02_silver.ipynb), puis les publier dans RustFS et PostgreSQL. Exécuter les cellules dans l'ordre avec les deux services démarrés.

### 1 - Préparer les connexions

Installer les dépendances, configurer l'accès S3 de DuckDB et créer le bucket Gold si nécessaire.

In [16]:
%pip install -q boto3 duckdb pandas sqlalchemy

Note: you may need to restart the kernel to use updated packages.


In [17]:
import os
from urllib.parse import urlsplit

import boto3
import duckdb
import pandas as pd
from botocore.config import Config
from sqlalchemy.engine import URL

endpoint = os.environ.get("S3_ENDPOINT", "http://rustfs:9000")
access_key = os.environ["S3_ACCESS_KEY"]
secret_key = os.environ["S3_SECRET_KEY"]
bucket_gold = "gold"

s3 = boto3.client(
    "s3",
    endpoint_url=endpoint,
    aws_access_key_id=access_key,
    aws_secret_access_key=secret_key,
    region_name="us-east-1",
    config=Config(signature_version="s3v4", s3={"addressing_style": "path"}),
)
if bucket_gold not in {bucket["Name"] for bucket in s3.list_buckets()["Buckets"]}:
    s3.create_bucket(Bucket=bucket_gold)

con = duckdb.connect(config={
    "memory_limit": os.getenv("GOLD_DUCKDB_MEMORY_LIMIT", "1GB"),
    "threads": 2,
})
con.execute("INSTALL httpfs")
con.execute("LOAD httpfs")

def literal_sql(valeur):
    return "'" + valeur.replace("'", "''") + "'"

s3_url = urlsplit(endpoint)
con.execute(f"""
    CREATE OR REPLACE SECRET rustfs_secret (
        TYPE S3,
        KEY_ID {literal_sql(access_key)},
        SECRET {literal_sql(secret_key)},
        ENDPOINT {literal_sql(s3_url.netloc)},
        URL_STYLE 'path',
        USE_SSL {s3_url.scheme == "https"}
    )
""")

def charger_postgres(lignes_exportees):
    url_postgres = URL.create(
        drivername="postgresql",
        username=os.environ["PG_USER"],
        password=os.environ["PG_PASSWORD"],
        host=os.environ["PG_HOST"],
        port=5432,
        database=os.environ["PG_DB"],
    ).render_as_string(hide_password=False)

    con.execute("INSTALL postgres").fetchall()
    con.execute("LOAD postgres").fetchall()
    con.execute(f"ATTACH {literal_sql(url_postgres)} AS postgres_gold (TYPE POSTGRES)").fetchall()

    bilan = []
    try:
        for nom_table, nombre_exporte in lignes_exportees.items():
            print(f"Chargement de {nom_table}", flush=True)
            table = f'postgres_gold.public."{nom_table}"'
            source = literal_sql(f"s3://{bucket_gold}/{nom_table}.parquet")
            con.execute("BEGIN TRANSACTION").fetchall()
            try:
                con.execute(f"DROP TABLE IF EXISTS {table}").fetchall()
                con.execute(f"CREATE TABLE {table} AS SELECT * FROM read_parquet({source})").fetchall()
                [(nombre_charge,)] = con.execute(f"SELECT COUNT(*) FROM {table}").fetchall()
                if nombre_charge != nombre_exporte:
                    raise RuntimeError(
                        f"{nom_table} : {nombre_charge} lignes chargées, {nombre_exporte} attendues."
                    )
                con.execute("COMMIT").fetchall()
            except Exception:
                con.execute("ROLLBACK").fetchall()
                raise

            bilan.append((nom_table, nombre_exporte, nombre_charge))
    finally:
        con.execute("DETACH postgres_gold").fetchall()
    return bilan

### 2 - Nommer les colonnes

Les dictionnaires associent les champs Silver aux noms utilisés dans les tables d'analyse.

In [18]:
renommage_openprices = {
    "id": "id_produit",
    "code": "code_barres",
    "source": "source_donnees",
    "source_last_synced": "date_derniere_synchronisation",
    "product_name": "nom_produit",
    "image_url": "url_image",
    "product_quantity": "quantite_produit",
    "product_quantity_unit": "unite_quantite_produit",
    "quantity": "quantite",
    "categories_tags": "categories",
    "brands": "marques",
    "brands_tags": "marques_tags",
    "labels_tags": "labels",
    "nutriscore_grade": "nutriscore",
    "ecoscore_grade": "ecoscore",
    "nova_group": "groupe_nova",
    "creator": "createur",
    "unique_scans_n": "nombre_scans_uniques",
    "price_count": "nombre_observations_prix",
    "price_currency_count": "nombre_devises_prix",
    "location_count": "nombre_lieux",
    "location_type_osm_country_count": "nombre_pays_lieux",
    "user_count": "nombre_utilisateurs",
    "proof_count": "nombre_justificatifs",
    "created": "date_creation",
    "updated": "date_mise_a_jour"
}

renommage_openfoodfacts = {
    "code": "code_barres",
    "product_name": "nom_produit",
    "brands": "marques",
    "categories_tags": "categories",
    "allergens_tags": "allergenes",
    "ingredients_text": "liste_ingredients",
    "nova_group": "groupe_nova",
    "nutriscore_grade": "nutriscore",
    "serving_quantity": "quantite_portion",
    "nutriments": "nutriments",
    "nutriments_estimated": "nutriments_estimes",
    "nutrition_data": "donnees_nutritionnelles",
    "nutrition_data_per": "donnees_nutritionnelles_pourc",
    "nutrition_data_prepared": "donnees_nutritionnelles_preparees",
    "nutrition_data_prepared_per": "donnees_nutritionnelles_preparees_pourc"
}

renommage_prix = {
    "id": "id_prix",
    "product_id": "id_produit",
    "location_id": "id_lieu",
    "proof_id": "id_justificatif",
    "type": "type_prix",
    "product_code": "code_barres",
    "product_name": "nom_produit",
    "category_tag": "categorie",
    "labels_tags": "labels",
    "origins_tags": "origines",
    "price": "prix",
    "price_is_discounted": "prix_en_promotion",
    "price_without_discount": "prix_hors_promotion",
    "discount_type": "type_promotion",
    "price_per": "unite_prix",
    "currency": "devise",
    "location_osm_id": "id_osm_lieu",
    "location_osm_type": "type_osm_lieu",
    "date": "date_prix",
    "receipt_quantity": "quantite_ticket",
    "owner_comment": "commentaire_proprietaire",
    "owner": "proprietaire",
    "source": "source_donnees",
    "tags": "tags",
    "created": "date_creation",
    "updated": "date_mise_a_jour",
    "duplicate_of": "id_prix_doublon"
}

renommage_lieux = {
    "id": "id_lieu",
    "osm_brand_logo_url": "url_logo_marque",
    "type": "type_lieu",
    "osm_id": "id_osm",
    "osm_type": "type_osm",
    "osm_name": "nom_osm",
    "osm_display_name": "nom_affichage",
    "osm_tag_key": "cle_tag_osm",
    "osm_tag_value": "valeur_tag_osm",
    "osm_tags": "tags_osm",
    "osm_brand": "marque",
    "osm_address_postcode": "code_postal",
    "osm_address_city": "ville",
    "osm_address_country": "pays",
    "osm_address_country_code": "code_pays",
    "osm_lat": "latitude",
    "osm_lon": "longitude",
    "osm_version": "version_osm",
    "osm_version_date": "date_version_osm",
    "website_url": "url_site_web",
    "tags": "tags",
    "price_count": "nombre_prix",
    "user_count": "nombre_utilisateurs",
    "product_count": "nombre_produits",
    "proof_count": "nombre_justificatifs",
    "source": "source_donnees",
    "created": "date_creation",
    "updated": "date_mise_a_jour"
}

renommage_justificatifs = {
    "id": "id_justificatif",
    "location_id": "id_lieu",
    "file_path": "chemin_fichier",
    "mimetype": "type_mime",
    "type": "type_justificatif",
    "image_md5_hash": "empreinte_md5_image",
    "image_thumb_path": "chemin_miniature",
    "location_osm_id": "id_osm_lieu",
    "location_osm_type": "type_osm_lieu",
    "date": "date_justificatif",
    "currency": "devise",
    "receipt_price_count": "nombre_prix_ticket",
    "receipt_price_total": "montant_total_ticket",
    "receipt_online_delivery_costs": "frais_livraison",
    "ready_for_price_tag_validation": "pret_validation_prix",
    "owner_consumption": "consommation_proprietaire",
    "owner_comment": "commentaire_proprietaire",
    "draft": "brouillon",
    "price_count": "nombre_prix",
    "prediction_count": "nombre_predictions",
    "owner": "proprietaire",
    "source": "source_donnees",
    "tags": "tags",
    "created": "date_creation",
    "updated": "date_mise_a_jour"
}

renommage_commandes = {
    "order_id": "id_commande",
    "user_id": "id_utilisateur",
    "eval_set": "ensemble_evaluation",
    "order_number": "numero_commande",
    "order_dow": "jour_semaine_commande",
    "order_hour_of_day": "heure_commande",
    "days_since_prior_order": "jours_depuis_commande_precedente"
}

renommage_lignes_commandes = {
    "order_id": "id_commande",
    "product_id": "id_produit_instacart",
    "add_to_cart_order": "ordre_ajout_panier",
    "reordered": "deja_commande"
}

renommage_produits_instacart = {
    "product_id": "id_produit_instacart",
    "product_name": "nom_produit",
    "aisle_id": "id_rayon",
    "department_id": "id_departement"
}

renommage_rayons = {
    "aisle_id": "id_rayon",
    "aisle": "nom_rayon"
}

renommage_departements = {
    "department_id": "id_departement",
    "department": "nom_departement"
}

renommage_rappels = {
    "categorie_produit": "categorie_produit",
    "conditionnements": "conditionnements",
    "conduites_a_tenir_par_le_consommateur": "consignes_consommateur",
    "date_date_fin_commercialisation": "date_fin_commercialisation",
    "date_de_fin_de_la_procedure_de_rappel": "date_fin_procedure_rappel",
    "date_debut_commercialisation": "date_debut_commercialisation",
    "date_publication": "date_publication",
    "description_complementaire_risque": "description_risque",
    "distributeurs": "distributeurs",
    "id": "id_rappel",
    "identification_produits": "identification_produits",
    "informations_complementaires": "informations_complementaires",
    "informations_complementaires_publiques": "informations_publiques",
    "libelle": "libelle",
    "lien_vers_affichette_pdf": "url_affichette_pdf",
    "lien_vers_la_fiche_rappel": "url_fiche_rappel",
    "lien_vers_la_liste_des_distributeurs": "url_liste_distributeurs",
    "lien_vers_la_liste_des_produits": "url_liste_produits",
    "liens_vers_les_images": "url_images",
    "marque_produit": "marque_produit",
    "marque_salubrite": "marque_salubrite",
    "modalites_de_compensation": "modalites_compensation",
    "modeles_ou_references": "modeles_references",
    "motif_rappel": "motif_rappel",
    "nature_juridique_rappel": "nature_juridique_rappel",
    "numero_contact": "numero_contact",
    "numero_fiche": "numero_fiche",
    "numero_version": "numero_version",
    "preconisations_sanitaires": "preconisations_sanitaires",
    "rappel_guid": "identifiant_rappel_guid",
    "risques_encourus": "risques_encourus",
    "sous_categorie_produit": "sous_categorie_produit",
    "temperature_conservation": "temperature_conservation",
    "zone_geographique_de_vente": "zone_geographique_vente"
}

### 3 - Open Prices et Open Food Facts

Construire le catalogue, la dimension des lieux de vente et celle des justificatifs, puis enrichir les observations de prix.

#### Catalogue de produits

Enrichir le catalogue Open Prices avec les données Open Food Facts à partir du code-barres.

In [19]:
produits_openprices = con.execute("""
    SELECT *
    FROM read_parquet('s3://silver/openprices/products.parquet')
""").fetchdf()

produits_openprices = produits_openprices.rename(
    columns=renommage_openprices
)

produits_openfoodfacts = con.execute("""
    SELECT *
    FROM read_parquet('s3://silver/openfoodfacts/produits.parquet')
""").fetchdf()

produits_openfoodfacts = produits_openfoodfacts.rename(
    columns=renommage_openfoodfacts
)

dim_produits = produits_openprices.merge(
    produits_openfoodfacts,
    on="code_barres",
    how="left",
    suffixes=("", "_off"),
    indicator=True
)

dim_produits["enrichi_openfoodfacts"] = (
    dim_produits["_merge"] == "both"
)

dim_produits = dim_produits.drop(columns=["_merge"])

#### Lieux et justificatifs

Préparer les dimensions qui décrivent les points de vente et les preuves associées aux relevés.

In [20]:
lieux_vente = con.execute("""
    SELECT *
    FROM read_parquet('s3://silver/openprices/locations.parquet')
""").fetchdf()

dim_lieux_vente = lieux_vente.rename(columns=renommage_lieux)

justificatifs_openprices = con.execute("""
    SELECT *
    FROM read_parquet('s3://silver/openprices/proofs.parquet')
""").fetchdf()

dim_justificatifs = justificatifs_openprices.rename(
    columns=renommage_justificatifs
)

#### Observations de prix

Retrouver les identifiants produits manquants par code-barres, puis ajouter les attributs du catalogue et du lieu. Les jointures vérifient que les dimensions ne multiplient pas les observations.

In [21]:
observations_prix = con.execute("""
    SELECT *
    FROM read_parquet('s3://silver/openprices/prices.parquet')
""").fetchdf()

fait_observations_prix = observations_prix.rename(
    columns=renommage_prix
)

fait_observations_prix["id_produit_source"] = (
    fait_observations_prix["id_produit"]
)

correspondance_codes = (
    dim_produits
    .dropna(subset=["code_barres"])
    .drop_duplicates(subset=["code_barres"])
    .set_index("code_barres")["id_produit"]
)

identifiants_correspondants = (
    fait_observations_prix["code_barres"]
    .map(correspondance_codes)
)

fait_observations_prix["produit_rattache_par_code_barres"] = (
    fait_observations_prix["id_produit"].isna() &
    identifiants_correspondants.notna()
)

fait_observations_prix["id_produit"] = (
    fait_observations_prix["id_produit"]
    .fillna(identifiants_correspondants)
)

colonnes_produits = [
    "id_produit",
    "nom_produit",
    "marques",
    "categories",
    "labels",
    "nutriscore",
    "ecoscore",
    "groupe_nova",
    "enrichi_openfoodfacts"
]

fait_observations_prix = fait_observations_prix.merge(
    dim_produits[colonnes_produits],
    on="id_produit",
    how="left",
    suffixes=("", "_catalogue"),
    validate="many_to_one"
)

colonnes_lieux = [
    "id_lieu",
    "marque",
    "nom_osm",
    "nom_affichage",
    "type_lieu",
    "cle_tag_osm",
    "valeur_tag_osm",
    "code_postal",
    "ville",
    "pays",
    "latitude",
    "longitude"
]

fait_observations_prix = fait_observations_prix.merge(
    dim_lieux_vente[colonnes_lieux],
    on="id_lieu",
    how="left",
    suffixes=("", "_lieu"),
    validate="many_to_one"
)

### 4 - Instacart

Préparer les référentiels et les commandes, puis réunir les lignes `prior` et `train` en conservant leur groupe d'origine.

In [22]:
produits_instacart = con.execute("""
    SELECT *
    FROM read_parquet('s3://silver/instacart/products.parquet')
""").fetchdf()

dim_produits_instacart = produits_instacart.rename(
    columns=renommage_produits_instacart
)

rayons_instacart = con.execute("""
    SELECT *
    FROM read_parquet('s3://silver/instacart/aisles.parquet')
""").fetchdf()

dim_rayons = rayons_instacart.rename(
    columns=renommage_rayons
)

departements_instacart = con.execute("""
    SELECT *
    FROM read_parquet('s3://silver/instacart/departments.parquet')
""").fetchdf()

dim_departements = departements_instacart.rename(
    columns=renommage_departements
)

In [23]:
def projection_gold(renommage):
    return ", ".join(f'"{source}" AS "{cible}"' for source, cible in renommage.items())

requete_commandes = f"""
    SELECT {projection_gold(renommage_commandes)}
    FROM read_parquet('s3://silver/instacart/orders.parquet')
"""

projection_lignes = projection_gold(renommage_lignes_commandes)
requete_lignes_commandes = f"""
    SELECT {projection_lignes}, 'prior' AS ensemble_evaluation
    FROM read_parquet('s3://silver/instacart/order_products_prior.parquet')
    UNION ALL
    SELECT {projection_lignes}, 'train' AS ensemble_evaluation
    FROM read_parquet('s3://silver/instacart/order_products_train.parquet')
"""

### 5 - RappelConso

Préparer la table des rappels à partir des champs de Silver.

In [24]:
rappels = con.execute("""
    SELECT *
    FROM read_parquet('s3://silver/rappelconso/rappels.parquet')
""").fetchdf()

fait_rappels = rappels.rename(
    columns=renommage_rappels
)

### 6 - Publier les tables dans RustFS

Écrire les dix tables en Parquet compressé. DuckDB exporte les commandes et les lignes de commande directement depuis Silver. Comparer le nombre de lignes écrites à celui du fichier stocké avant de charger PostgreSQL.

In [25]:
tables_gold = {
    "dim_rayons": dim_rayons,
    "dim_departements": dim_departements,
    "dim_produits_instacart": dim_produits_instacart,
    "dim_produits": dim_produits,
    "dim_lieux_vente": dim_lieux_vente,
    "fait_observations_prix": fait_observations_prix,
    "dim_justificatifs": dim_justificatifs,
    "fait_rappels": fait_rappels,
}
requetes_gold = {
    "fait_commandes": requete_commandes,
    "fait_lignes_commandes": requete_lignes_commandes,
}

lignes_exportees = {}
with con.cursor() as con_export:
    for nom_table, dataframe in tables_gold.items():
        con_export.register(nom_table, dataframe)
    for nom_table, requete in requetes_gold.items():
        con_export.execute(f'CREATE TEMP VIEW "{nom_table}" AS {requete}')

    for nom_table in (*tables_gold, *requetes_gold):
        destination = literal_sql(f"s3://{bucket_gold}/{nom_table}.parquet")
        print(f"Export de {nom_table}", flush=True)
        [(nombre_ecrit,)] = con_export.execute(f"""
            COPY (SELECT * FROM "{nom_table}") TO {destination}
            (FORMAT PARQUET, COMPRESSION ZSTD, OVERWRITE_OR_IGNORE TRUE)
        """).fetchall()
        [(nombre_stocke,)] = con_export.execute(
            f"SELECT COUNT(*) FROM read_parquet({destination})"
        ).fetchall()
        if nombre_stocke != nombre_ecrit:
            raise RuntimeError(
                f"{nom_table} : {nombre_stocke} lignes stockées, {nombre_ecrit} écrites."
            )
        lignes_exportees[nom_table] = nombre_stocke

Export de dim_rayons
Export de dim_departements
Export de dim_produits_instacart
Export de dim_produits
Export de dim_lieux_vente
Export de fait_observations_prix
Export de dim_justificatifs
Export de fait_rappels
Export de fait_commandes
Export de fait_lignes_commandes


### 7 - Charger les tables dans PostgreSQL

Remplacer les tables du schéma `public` par les données des fichiers Gold. Le tableau final compare les nombres de lignes dans RustFS et PostgreSQL.

In [26]:
bilan = charger_postgres(lignes_exportees)

display(pd.DataFrame(bilan, columns=["Table Gold", "Lignes RustFS", "Lignes PostgreSQL"]))

Chargement de dim_rayons
Chargement de dim_departements
Chargement de dim_produits_instacart
Chargement de dim_produits
Chargement de dim_lieux_vente
Chargement de fait_observations_prix


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Chargement de dim_justificatifs
Chargement de fait_rappels
Chargement de fait_commandes
Chargement de fait_lignes_commandes


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,Table Gold,Lignes RustFS,Lignes PostgreSQL
0,dim_rayons,134,134
1,dim_departements,21,21
2,dim_produits_instacart,49688,49688
3,dim_produits,150354,150354
4,dim_lieux_vente,7947,7947
5,fait_observations_prix,345806,345806
6,dim_justificatifs,134585,134585
7,fait_rappels,19260,19260
8,fait_commandes,3421083,3421083
9,fait_lignes_commandes,33819106,33819106


### 8 - Joindre les lignes de commande et les commandes Instacart

Enrichir chaque ligne de commande avec les informations de sa commande via `id_commande`. La jointure gauche conserve toutes les lignes de départ. Les commandes sans ligne sont disponibles dans `fait_commandes`.

DuckDB écrit le résultat directement en Parquet dans RustFS, puis le publie dans PostgreSQL sous `public.fait_lignes_commandes_enrichies`.

Si les deux Parquet Instacart Gold existent déjà, exécuter la cellule de connexion puis les deux cellules ci-dessous.

In [27]:
fichier_lignes = literal_sql(f"s3://{bucket_gold}/fait_lignes_commandes.parquet")
fichier_commandes = literal_sql(f"s3://{bucket_gold}/fait_commandes.parquet")
destination_jointure = literal_sql(f"s3://{bucket_gold}/fait_lignes_commandes_enrichies.parquet")

(nombre_commandes, nombre_ids) = con.execute(f"""
    SELECT COUNT(*), COUNT(DISTINCT id_commande)
    FROM read_parquet({fichier_commandes})
""").fetchone()
if nombre_commandes != nombre_ids:
    raise RuntimeError("fait_commandes doit contenir un identifiant unique et renseigné par commande.")

(nombre_lignes_source,) = con.execute(
    f"SELECT COUNT(*) FROM read_parquet({fichier_lignes})"
).fetchone()

requete_jointure = f"""
    SELECT l.*,
           c.id_utilisateur, c.numero_commande, c.jour_semaine_commande,
           c.heure_commande, c.jours_depuis_commande_precedente
    FROM read_parquet({fichier_lignes}) l
    LEFT JOIN read_parquet({fichier_commandes}) c USING (id_commande)
"""
print("Export des lignes de commande enrichies", flush=True)
(nombre_ecrit,) = con.execute(f"""
    COPY ({requete_jointure}) TO {destination_jointure}
    (FORMAT PARQUET, COMPRESSION ZSTD, OVERWRITE_OR_IGNORE TRUE)
""").fetchone()
(nombre_stocke,) = con.execute(
    f"SELECT COUNT(*) FROM read_parquet({destination_jointure})"
).fetchone()
if nombre_ecrit != nombre_lignes_source or nombre_stocke != nombre_lignes_source:
    raise RuntimeError(
        f"{nombre_lignes_source} lignes attendues, {nombre_ecrit} écrites, {nombre_stocke} stockées."
    )

print(f"{nombre_stocke:,} lignes dans fait_lignes_commandes_enrichies.")
display(con.execute(f"SELECT * FROM read_parquet({destination_jointure}) LIMIT 20").fetchdf())

Export des lignes de commande enrichies


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

33,819,106 lignes dans fait_lignes_commandes_enrichies.


,id_commande,id_produit_instacart,ordre_ajout_panier,deja_commande,ensemble_evaluation,id_utilisateur,numero_commande,jour_semaine_commande,heure_commande,jours_depuis_commande_precedente
0,251350,26683,5,1,prior,94948,5,2,22,14.0
1,251350,37197,6,0,prior,94948,5,2,22,14.0
2,251350,33198,7,0,prior,94948,5,2,22,14.0
3,251350,43704,8,0,prior,94948,5,2,22,14.0
4,251350,20114,9,0,prior,94948,5,2,22,14.0
5,251350,36070,10,0,prior,94948,5,2,22,14.0
6,251350,3494,11,0,prior,94948,5,2,22,14.0
7,251350,24852,12,1,prior,94948,5,2,22,14.0
8,251350,18377,13,0,prior,94948,5,2,22,14.0
9,251350,32163,14,0,prior,94948,5,2,22,14.0


In [28]:
nom_table_jointure = "fait_lignes_commandes_enrichies"
fichier_jointure = literal_sql(f"s3://{bucket_gold}/{nom_table_jointure}.parquet")
(nombre_exporte_jointure,) = con.execute(
    f"SELECT COUNT(*) FROM read_parquet({fichier_jointure})"
).fetchone()

bilan_jointure = charger_postgres({nom_table_jointure: nombre_exporte_jointure})
display(pd.DataFrame(bilan_jointure, columns=["Table Gold", "Lignes RustFS", "Lignes PostgreSQL"]))

Chargement de fait_lignes_commandes_enrichies


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,Table Gold,Lignes RustFS,Lignes PostgreSQL
0,fait_lignes_commandes_enrichies,33819106,33819106
